# CFEVER：三階段事實查核（改進版）

本 notebook 以助教範例程式為基礎，三個階段都改寫過：

1. **Document Retrieval**：在**完整** Wikipedia（約 119 萬頁）上做**標題比對**，再用啟發式分數排序候選頁面。
2. **Evidence Retrieval**：取前 5 頁的所有句子，先用字元 bigram 重疊度排序作為基準，
   再以 **cross-encoder（MacBERT-base）** 重新排序，保留前 5 句。
3. **Classification**：以 **RoBERTa-wwm-ext-large** 讀取 `[CLS] claim [SEP] 證據 [SEP]`，預測三類標籤。

> 範例程式的檢索語料只收「訓練集證據涉及的頁面」，但 dev 的標準證據頁只有約 2.5% 落在這些頁面裡，
> 所以這裡改為索引完整的 `wiki_pages.jsonl`。

**Generative AI 使用聲明**：本 notebook 的程式碼與註解由 Claude（Anthropic）協助撰寫，
以助教提供的 sample code 為基礎修改；方法設計、本機驗證與結果判讀由我確認。

## 1. 套件與參數

請在 Kaggle 啟用 **GPU（T4）** 並開啟 **Internet**（第一次需要從 Hugging Face 下載模型）。

In [1]:
# [Generative AI] 本 cell 由 Claude 協助撰寫：匯入套件、固定亂數種子。
import gc
import json
import math
import os
import random
import re
import time
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

SEED = 2026


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"  # T4 上用 fp16 混合精度加速
print("使用裝置：", device)

使用裝置： cuda


In [2]:
# [Generative AI] 本 cell 由 Claude 協助撰寫：路徑與超參數。
if Path("/kaggle/input").exists():
    DATA_DIR = Path("/kaggle/input")
    OUTPUT_DIR = Path("/kaggle/working")
else:
    DATA_DIR = Path(os.environ.get("HW3_DATA_DIR", "."))
    OUTPUT_DIR = Path("hw3_output")


def find_data_file(filename):
    folder = Path(DATA_DIR)
    direct_path = folder / filename
    if direct_path.is_file():
        return direct_path
    matches = sorted(path for path in folder.rglob(filename) if path.is_file())
    if not matches:
        raise FileNotFoundError(f"找不到 {filename}。請加入競賽資料，或修改 DATA_DIR。目前搜尋目錄：{folder}")
    if len(matches) > 1:
        paths = "\n".join(str(path) for path in matches)
        raise ValueError(f"找到多份 {filename}，請把 DATA_DIR 改成指定的資料夾：\n{paths}")
    return matches[0]


TRAIN_PATH = find_data_file("train.jsonl")
DEV_PATH = find_data_file("dev.jsonl")
TEST_PATH = find_data_file("test.jsonl")
WIKI_PATH = find_data_file("wiki_pages.jsonl")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(TRAIN_PATH, DEV_PATH, TEST_PATH, WIKI_PATH, sep="\n")

# 本機煙霧測試用：設定環境變數 HW3_SMOKE=1 會改用極小的模型與少量資料。Kaggle 上不會啟用。
SMOKE_TEST = os.environ.get("HW3_SMOKE") == "1"

DOCUMENT_TOP_K = 5        # 第一階段評估用的頁面數（Recall@5）
EVIDENCE_PAGE_K = 8       # 第二階段實際從前幾頁挑句子（多給 cross-encoder 一些候選）
ALIAS_WEIGHT = -0.5       # 只靠別名比對到的頁面略為扣分（dev 上調得）
MAX_TITLE_LENGTH = 30     # 標題比對時，claim 子字串的最大長度
EVIDENCE_TOP_K = 5        # 第二階段保留的句子數（metric 只看前 5 句）

RERANK_MODEL_NAME = "hfl/chinese-macbert-base"
RERANK_MAX_LENGTH = 128
RERANK_BATCH_SIZE = 32
RERANK_EPOCHS = 1
RERANK_LR = 3e-5
RERANK_HARD_NEGATIVES = 5   # 每個 claim 取 bigram 排名高、但不是標準答案的句子
RERANK_RANDOM_NEGATIVES = 3 # 每個 claim 從候選句中隨機抽的負例

CLS_MODEL_NAME = "hfl/chinese-roberta-wwm-ext-large"
CLS_MAX_LENGTH = 256
CLS_BATCH_SIZE = 8
CLS_GRAD_ACCUM = 2          # 等效 batch size = 16
CLS_EPOCHS = 2
CLS_LR = 1e-5
WARMUP_RATIO = 0.1

TRAIN_SAMPLES = None  # None 表示使用全部資料
DEV_SAMPLES = None
TEST_SAMPLES = None
if SMOKE_TEST:
    RERANK_MODEL_NAME = CLS_MODEL_NAME = os.environ.get("HW3_TINY_MODEL", "ckiplab/bert-tiny-chinese")
    TRAIN_SAMPLES, DEV_SAMPLES, TEST_SAMPLES = 400, 200, 100
    CLS_EPOCHS = 1

/kaggle/input/competitions/nthu-nlp-2026-assignment-3-fact-verification/train.jsonl
/kaggle/input/competitions/nthu-nlp-2026-assignment-3-fact-verification/dev.jsonl
/kaggle/input/competitions/nthu-nlp-2026-assignment-3-fact-verification/test.jsonl
/kaggle/input/competitions/nthu-nlp-2026-assignment-3-fact-verification/wiki_pages.jsonl


### 讀取資料

這次使用**全部**訓練與驗證資料（範例只抽 3000／300 筆）。

In [3]:
# [Generative AI] 本 cell 由 Claude 協助撰寫：讀取 JSONL。
def read_jsonl(path):
    with Path(path).open(encoding="utf-8-sig") as file:
        return pd.read_json(file, lines=True)


train_df = read_jsonl(TRAIN_PATH)
dev_df = read_jsonl(DEV_PATH)
test_df = read_jsonl(TEST_PATH)[["id", "claim"]]  # 測試集只使用輸入欄位

if TRAIN_SAMPLES is not None:
    train_df = train_df.sample(n=min(TRAIN_SAMPLES, len(train_df)), random_state=SEED)
if DEV_SAMPLES is not None:
    dev_df = dev_df.sample(n=min(DEV_SAMPLES, len(dev_df)), random_state=SEED)
if TEST_SAMPLES is not None:
    test_df = test_df.head(TEST_SAMPLES)

train_df = train_df.reset_index(drop=True)
dev_df = dev_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
for df in (train_df, dev_df):
    df["label"] = df["label"].str.upper()

print("訓練／驗證／測試筆數：", len(train_df), len(dev_df), len(test_df))
print(train_df["label"].value_counts())

訓練／驗證／測試筆數： 24012 3000 6000
label
SUPPORTS           11085
REFUTES             7113
NOT ENOUGH INFO     5814
Name: count, dtype: int64


## 2. Document Retrieval：標題比對

觀察：CFEVER 的 claim 是標註者改寫 Wikipedia 句子而來，**頁面標題通常直接出現在 claim 中**。
在 dev 上，約 87% 可驗證題目的所有證據頁標題（去掉 `_(消歧義)` 括號後）都是 claim 的子字串。

做法：

1. 掃一次 Wikipedia，建立「正規化標題 → 頁面名稱」的字典。正規化會去掉結尾的 `_(...)`、移除空白與底線、統一間隔號
   （claim 常寫「尼爾斯 · 索爾森」或 `Super_Junior`，與標題寫法不同）。
2. 列舉 claim 中長度 1～30 的所有子字串，查字典取得候選頁面。
3. 對每個候選頁面算啟發式分數，取前 5 名：
   - 標題長度：越長越具體（單字標題如「美」通常是雜訊）；
   - 若比對到的片段被另一個更長的候選標題完全涵蓋，就扣分（例如「芝加哥大學」涵蓋「芝加哥」）；
   - claim 與頁面開頭 150 字、前 1000 字的字元 bigram 重疊率，可用來分辨同名消歧義頁；
   - 帶有 `_(...)` 的消歧義頁面略為扣分。
   - 權重以 dev 網格搜尋決定。

In [4]:
# [Generative AI] 本 cell 由 Claude 協助撰寫：第一次掃描 Wikipedia，只建立標題索引。
def base_title(title):
    """顯示用標題：去掉結尾的 _(消歧義說明)。"""
    return re.sub(r"_\(.*\)$", "", title).replace("_", " ")


DOT_PATTERN = re.compile(r"[·•‧・∙⋅]")


def normalize(text):
    """比對用正規化：移除空白與底線、統一間隔號。claim 常寫成「尼爾斯 · 索爾森」，標題則是「尼爾斯·索爾森」。"""
    return DOT_PATTERN.sub("·", re.sub(r"[\s_]+", "", text))


def title_key(title):
    return normalize(re.sub(r"_\(.*\)$", "", title))


def clean_text(text):
    """Wikipedia 文字在中文與標點之間有多餘空白；只移除緊鄰非 ASCII 字元的空白，保留英文單字間距。"""
    text = re.sub(r"\s+(?=[^\x00-\x7f])", "", text)
    text = re.sub(r"(?<=[^\x00-\x7f])\s+", "", text)
    return text.strip()


# 從頁面開頭擷取別名，例如「威尼斯電影節（又稱威尼斯影展）」，處理 claim 用別名或不同譯名的情況。
ALIAS_PATTERN = re.compile(
    r"(?:又稱|又名|簡稱|或稱|舊稱|亦稱|也稱|通稱|俗稱|別稱|又譯|或譯|譯作|譯爲|全稱|原名|本名)[爲作]?[：:]?「?《?"
    r"([^，。、；（）「」《》()]{2,15})"
)

start_time = time.time()
title_index = defaultdict(list)
alias_index = defaultdict(set)
with WIKI_PATH.open(encoding="utf-8") as file:
    for line in file:
        page = json.loads(line)
        page_id = page["id"]
        key = title_key(page_id)
        if key:
            title_index[key].append(page_id)
        for match in ALIAS_PATTERN.finditer(normalize(page["text"][:300])):
            alias = match.group(1).strip("」》")
            if len(alias) >= 2:
                alias_index[alias].add(page_id)
# 和既有標題相同的別名不加入，避免重複與雜訊。
for alias in [a for a in alias_index if a in title_index]:
    del alias_index[alias]
print(f"標題數：{len(title_index)}，別名數：{len(alias_index)}，耗時 {time.time() - start_time:.0f} 秒")

標題數：1081060，別名數：117773，耗時 34 秒


In [5]:
# [Generative AI] 本 cell 由 Claude 協助撰寫：從 claim 找出所有標題比對候選。
def find_title_candidates(claim):
    """回傳 {頁面名稱: [(起點, 終點, 是否為別名), ...]}，記錄標題或別名在 claim 中出現的位置。"""
    candidates = {}
    claim = normalize(claim)
    length = len(claim)
    for start in range(length):
        for end in range(start + 1, min(length, start + MAX_TITLE_LENGTH) + 1):
            piece = claim[start:end]
            for page_id in title_index.get(piece, ()):
                candidates.setdefault(page_id, []).append((start, end, False))
            if end - start >= 2:
                for page_id in alias_index.get(piece, ()):
                    candidates.setdefault(page_id, []).append((start, end, True))
    return candidates


train_candidates = [find_title_candidates(c) for c in train_df["claim"]]
dev_candidates = [find_title_candidates(c) for c in dev_df["claim"]]
test_candidates = [find_title_candidates(c) for c in test_df["claim"]]

# 第二次掃描只需要載入「候選頁面」以及「訓練集標準證據頁面」的內容。
needed_pages = set()
for candidate_list in (train_candidates, dev_candidates, test_candidates):
    for candidates in candidate_list:
        needed_pages.update(candidates)
for row in train_df.itertuples():
    if row.label != "NOT ENOUGH INFO":
        for group in row.evidence:
            for item in group:
                needed_pages.add(item[2])
print("需要載入的頁面數：", len(needed_pages))
print("平均候選頁面數（dev）：", np.mean([len(c) for c in dev_candidates]))

需要載入的頁面數： 48121
平均候選頁面數（dev）： 61.668


In [6]:
# [Generative AI] 本 cell 由 Claude 協助撰寫：第二次掃描，保留需要頁面的文字與句子（保留原始句子編號）。
page_text = {}    # 頁面名稱 -> 清理後的全文
page_lines = {}   # 頁面名稱 -> {句子編號: 句子}
start_time = time.time()
with WIKI_PATH.open(encoding="utf-8") as file:
    for line in file:
        page = json.loads(line)
        if page["id"] not in needed_pages:
            continue
        page_text[page["id"]] = clean_text(page["text"])
        sentences = {}
        for sentence_line in page["lines"].split("\n"):
            parts = sentence_line.split("\t")
            if len(parts) < 2 or not parts[0].isdigit():
                continue
            text = clean_text(parts[1])
            if text:
                sentences[int(parts[0])] = text
        page_lines[page["id"]] = sentences
print(f"已載入 {len(page_text)} 頁，耗時 {time.time() - start_time:.0f} 秒")

已載入 48121 頁，耗時 12 秒


In [7]:
# [Generative AI] 本 cell 由 Claude 協助撰寫：候選頁面排序。
def char_bigrams(text):
    return {text[i:i + 2] for i in range(len(text) - 1)}


def rank_documents(claim, candidates, top_k):
    claim_bigrams = char_bigrams(normalize(claim))
    all_spans = [(span, page_id) for page_id, spans in candidates.items() for span in spans]
    scores = {}
    for page_id, spans in candidates.items():
        title_length = max(end - start for start, end, _ in spans)
        alias_only = all(is_alias for _, _, is_alias in spans)
        # 比對片段是否被更長的其他標題完全涵蓋
        covered = any(
            other_start <= start and end <= other_end and (other_end - other_start) > (end - start)
            for (start, end, _) in spans
            for (other_start, other_end, _), _ in all_spans
        )
        text = normalize(page_text.get(page_id, ""))
        overlap_lead = len(claim_bigrams & char_bigrams(text[:150])) / max(len(claim_bigrams), 1)
        overlap_body = len(claim_bigrams & char_bigrams(text[:1000])) / max(len(claim_bigrams), 1)
        # 權重以 dev 上的網格搜尋決定（Recall@5：0.8605 → 0.878）
        scores[page_id] = (
            2.0 * (title_length >= 2)
            + 0.3 * min(title_length, 6)
            - 1.0 * covered
            + 5.0 * overlap_body
            + 2.0 * overlap_lead
            - 0.5 * ("_(" in page_id)
            + ALIAS_WEIGHT * alias_only
        )
    return sorted(scores, key=lambda page_id: -scores[page_id])[:top_k]


train_documents = [rank_documents(c, cand, EVIDENCE_PAGE_K) for c, cand in zip(train_df["claim"], train_candidates)]
dev_documents = [rank_documents(c, cand, EVIDENCE_PAGE_K) for c, cand in zip(dev_df["claim"], dev_candidates)]
test_documents = [rank_documents(c, cand, EVIDENCE_PAGE_K) for c, cand in zip(test_df["claim"], test_candidates)]

print("測試 claim：", test_df.loc[0, "claim"])
print("取回頁面：", test_documents[0])

測試 claim： 沈懋孝爲明朝的平民，對書籍厭惡，毀壞數量達萬卷。
取回頁面： ['沈懋孝', '明朝', '平民', '厭惡', '平民_(消歧義)', '平民_(古羅馬)', '標量_(物理學)', '明_(佛教)']


### Document Retrieval Recall@5

與範例相同的定義：只看 supports／refutes，前 5 頁必須涵蓋任一完整證據組的所有頁面。
另外印出「所有標題候選」的召回上限，用來分析錯誤來自候選生成還是排序。

In [8]:
# [Generative AI] 本 cell 由 Claude 協助撰寫：文件檢索評估。
def gold_groups(row, as_sentences):
    groups = []
    for group in row.evidence:
        if as_sentences:
            groups.append({(item[2], item[3]) for item in group})
        else:
            groups.append({item[2] for item in group})
    return groups


def recall_at(predictions, df, as_sentences, k=5):
    hits = count = 0
    for prediction, row in zip(predictions, df.itertuples()):
        if row.label == "NOT ENOUGH INFO":
            continue
        count += 1
        predicted = set(map(tuple, prediction[:k])) if as_sentences else set(prediction[:k])
        if any(group <= predicted for group in gold_groups(row, as_sentences)):
            hits += 1
    return hits / max(count, 1)


document_recall_at5 = recall_at(dev_documents, dev_df, as_sentences=False)
candidate_upper_bound = recall_at([list(c) for c in dev_candidates], dev_df, as_sentences=False, k=10**9)
print(f"Document Retrieval Recall@5: {document_recall_at5:.4f}")
print(f"標題候選召回上限（不限 K）：{candidate_upper_bound:.4f}")
for k in (1, 3, 10):
    deeper = [rank_documents(c, cand, k) for c, cand in zip(dev_df["claim"], dev_candidates)]
    print(f"Document Retrieval Recall@{k}: {recall_at(deeper, dev_df, as_sentences=False, k=k):.4f}")

Document Retrieval Recall@5: 0.8825
標題候選召回上限（不限 K）：0.8875
Document Retrieval Recall@1: 0.8160
Document Retrieval Recall@3: 0.8715
Document Retrieval Recall@10: 0.8855


## 3. Evidence Retrieval：bigram 粗排 + cross-encoder 重排

候選句子是前 5 頁的**所有**句子（範例只用訓練集標註過的句子，測試題的證據句大多不在其中）。

1. **基準**：claim 與句子的字元 bigram 重疊率，第 0 句（頁面開頭定義句）稍微加分。
2. **重排**：把 `claim` 與 `頁面標題：句子` 一起送入 MacBERT cross-encoder，輸出一個相關分數。
   - 正例：訓練集的標準證據句；
   - 困難負例：bigram 排名高、但不是標準證據的句子；
   - 隨機負例：從候選句中隨機抽取；
   - 以 Binary Cross Entropy 訓練。
3. **融合**：最終分數 = cross-encoder 分數 + λ × bigram 分數，λ 從 {0, 1, 3, 10, 只用 bigram} 中挑 dev Recall@5 最高者。

句子前面加上頁面標題，因為很多證據句用代名詞（「他」「該市」）指稱主題。

In [9]:
# [Generative AI] 本 cell 由 Claude 協助撰寫：建立候選句並計算 bigram 基準分數。
def build_sentence_candidates(documents):
    """每個 claim 回傳 [(頁面名稱, 句子編號, 句子), ...]。"""
    return [[(page_id, sid, text) for page_id in pages for sid, text in page_lines.get(page_id, {}).items()]
            for pages in documents]


def bigram_rank(claim, candidates):
    claim_bigrams = char_bigrams(claim)

    def score(candidate):
        _, sid, text = candidate
        return len(claim_bigrams & char_bigrams(text)) / max(len(claim_bigrams), 1) + (0.05 if sid == 0 else 0)

    return sorted(candidates, key=score, reverse=True)


train_sentences = build_sentence_candidates(train_documents)
dev_sentences = build_sentence_candidates(dev_documents)
test_sentences = build_sentence_candidates(test_documents)

dev_bigram = [bigram_rank(c, s) for c, s in zip(dev_df["claim"], dev_sentences)]
bigram_recall = recall_at([[[p, i] for p, i, _ in r] for r in dev_bigram], dev_df, as_sentences=True)
sentence_upper_bound = recall_at([[[p, i] for p, i, _ in s] for s in dev_sentences], dev_df, as_sentences=True, k=10**9)
print("dev 平均候選句數：", np.mean([len(s) for s in dev_sentences]))
print(f"Bigram 基準 Evidence Recall@5: {bigram_recall:.4f}")
print(f"候選句召回上限（受文件檢索限制）：{sentence_upper_bound:.4f}")

dev 平均候選句數： 53.75333333333333
Bigram 基準 Evidence Recall@5: 0.7490
候選句召回上限（受文件檢索限制）：0.8855


In [10]:
# [Generative AI] 本 cell 由 Claude 協助撰寫：建立 cross-encoder 訓練資料（正例、困難負例、隨機負例）。
def sentence_input(page_id, text):
    return f"{base_title(page_id)}：{text}"


rng = random.Random(SEED)
rerank_examples = []  # (claim, 句子輸入, 標籤 0/1)
for row, candidates in zip(train_df.itertuples(), train_sentences):
    positives = set()
    if row.label != "NOT ENOUGH INFO":
        for group in row.evidence:
            for item in group:
                positives.add((item[2], item[3]))
    for page_id, sid in positives:
        text = page_lines.get(page_id, {}).get(sid)
        if text:
            rerank_examples.append((row.claim, sentence_input(page_id, text), 1.0))
    negatives = [c for c in candidates if (c[0], c[1]) not in positives]
    hard = bigram_rank(row.claim, negatives)[:RERANK_HARD_NEGATIVES]
    rest = negatives and [c for c in negatives if c not in hard]
    random_negatives = rng.sample(rest, min(RERANK_RANDOM_NEGATIVES, len(rest))) if rest else []
    for page_id, sid, text in hard + random_negatives:
        rerank_examples.append((row.claim, sentence_input(page_id, text), 0.0))

rng.shuffle(rerank_examples)
print("重排訓練樣本數：", len(rerank_examples),
      "正例比例：", round(sum(e[2] for e in rerank_examples) / len(rerank_examples), 3))

重排訓練樣本數： 220423 正例比例： 0.129


In [11]:
# [Generative AI] 本 cell 由 Claude 協助撰寫：通用的訓練／推論工具（動態 padding、fp16、線性 warmup）。
def make_loader(tokenizer, firsts, seconds, labels, batch_size, max_length, shuffle, label_dtype):
    items = list(range(len(firsts)))

    def collate(indices):
        encoded = tokenizer([firsts[i] for i in indices], [seconds[i] for i in indices],
                            truncation="longest_first", max_length=max_length,
                            padding=True, return_tensors="pt")
        if labels is not None:
            encoded["labels"] = torch.tensor([labels[i] for i in indices], dtype=label_dtype)
        return encoded

    return DataLoader(items, batch_size=batch_size, shuffle=shuffle, collate_fn=collate)


def train_model(model, loader, epochs, lr, loss_fn, grad_accum=1, eval_fn=None):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = math.ceil(len(loader) / grad_accum) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    best_score, best_state, history = -1.0, None, []
    for epoch in range(epochs):
        model.train()
        running = 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(loader, start=1):
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop("labels")
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(**batch).logits
            loss = loss_fn(logits.float(), labels) / grad_accum
            scaler.scale(loss).backward()
            if step % grad_accum == 0 or step == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
            running += loss.item() * grad_accum
            if step % 200 == 0 or step == len(loader):
                print(f"epoch {epoch + 1} step {step}/{len(loader)} loss {running / step:.4f}")
        if eval_fn is not None:
            score = eval_fn()
            history.append(score)
            print(f"epoch {epoch + 1} dev 分數：{score:.4f}")
            if score > best_score:  # 保留 dev 最好的權重
                best_score = score
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return history


@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    outputs = []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            outputs.append(model(**batch).logits.float().cpu())
    return torch.cat(outputs, dim=0)

In [12]:
# [Generative AI] 本 cell 由 Claude 協助撰寫：訓練 cross-encoder 重排模型。
set_seed(SEED)
rerank_tokenizer = AutoTokenizer.from_pretrained(RERANK_MODEL_NAME)
rerank_model = AutoModelForSequenceClassification.from_pretrained(RERANK_MODEL_NAME, num_labels=1).to(device)

rerank_loader = make_loader(rerank_tokenizer, [e[0] for e in rerank_examples], [e[1] for e in rerank_examples],
                            [e[2] for e in rerank_examples], RERANK_BATCH_SIZE, RERANK_MAX_LENGTH,
                            shuffle=True, label_dtype=torch.float)
bce = lambda logits, labels: F.binary_cross_entropy_with_logits(logits.squeeze(-1), labels)
start_time = time.time()
train_model(rerank_model, rerank_loader, RERANK_EPOCHS, RERANK_LR, bce)
print(f"重排模型訓練耗時 {time.time() - start_time:.0f} 秒")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/269k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  412MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: hfl/chinese-macbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params 

epoch 1 step 200/6889 loss 0.4396
epoch 1 step 400/6889 loss 0.3295
epoch 1 step 600/6889 loss 0.2932
epoch 1 step 800/6889 loss 0.2712
epoch 1 step 1000/6889 loss 0.2579
epoch 1 step 1200/6889 loss 0.2505
epoch 1 step 1400/6889 loss 0.2421
epoch 1 step 1600/6889 loss 0.2373
epoch 1 step 1800/6889 loss 0.2320
epoch 1 step 2000/6889 loss 0.2282
epoch 1 step 2200/6889 loss 0.2239
epoch 1 step 2400/6889 loss 0.2190
epoch 1 step 2600/6889 loss 0.2157
epoch 1 step 2800/6889 loss 0.2126
epoch 1 step 3000/6889 loss 0.2107
epoch 1 step 3200/6889 loss 0.2081
epoch 1 step 3400/6889 loss 0.2060
epoch 1 step 3600/6889 loss 0.2036
epoch 1 step 3800/6889 loss 0.2015
epoch 1 step 4000/6889 loss 0.1994
epoch 1 step 4200/6889 loss 0.1975
epoch 1 step 4400/6889 loss 0.1953
epoch 1 step 4600/6889 loss 0.1937
epoch 1 step 4800/6889 loss 0.1920
epoch 1 step 5000/6889 loss 0.1902
epoch 1 step 5200/6889 loss 0.1889
epoch 1 step 5400/6889 loss 0.1872
epoch 1 step 5600/6889 loss 0.1859
epoch 1 step 5800/6889 l

In [13]:
# [Generative AI] 本 cell 由 Claude 協助撰寫：cross-encoder 打分，再與 bigram 分數融合（λ 在 dev 上挑選）。
def bigram_score(claim, sid, text):
    claim_bigrams = char_bigrams(claim)
    return len(claim_bigrams & char_bigrams(text)) / max(len(claim_bigrams), 1) + (0.05 if sid == 0 else 0)


def score_candidates(claims, sentence_lists):
    """回傳每個 claim 的 [(cross-encoder 分數, bigram 分數, 頁面, 句號, 句子), ...]。"""
    firsts, seconds = [], []
    for claim, candidates in zip(claims, sentence_lists):
        for page_id, sid, text in candidates:
            firsts.append(claim)
            seconds.append(sentence_input(page_id, text))
    scores = []
    if firsts:
        loader = make_loader(rerank_tokenizer, firsts, seconds, None, 128, RERANK_MAX_LENGTH,
                             shuffle=False, label_dtype=torch.float)
        scores = predict_logits(rerank_model, loader).squeeze(-1).tolist()
    results, position = [], 0
    for claim, candidates in zip(claims, sentence_lists):
        items = []
        for page_id, sid, text in candidates:
            items.append((scores[position], bigram_score(claim, sid, text), page_id, sid, text))
            position += 1
        results.append(items)
    return results


def select_top(scored, weight, top_k):
    """weight = None 表示只用 bigram；否則分數 = cross-encoder + weight × bigram。"""
    evidence, ranked = [], []
    for items in scored:
        key = (lambda x: -x[1]) if weight is None else (lambda x: -(x[0] + weight * x[1]))
        top = sorted(items, key=key)[:top_k]
        ranked.append(top)
        evidence.append([[page_id, sid] for _, _, page_id, sid, _ in top])
    return evidence, ranked


start_time = time.time()
dev_scored = score_candidates(dev_df["claim"].tolist(), dev_sentences)
test_scored = score_candidates(test_df["claim"].tolist(), test_sentences)
print(f"重排推論耗時 {time.time() - start_time:.0f} 秒")

fusion_results = {}
for weight in [0.0, 1.0, 3.0, 10.0, None]:
    evidence, _ = select_top(dev_scored, weight, EVIDENCE_TOP_K)
    fusion_results[weight] = recall_at(evidence, dev_df, as_sentences=True)
    print(f"λ = {weight}（None 表示只用 bigram）: Evidence Recall@5 = {fusion_results[weight]:.4f}")
best_weight = max(fusion_results, key=lambda w: fusion_results[w])
cross_encoder_recall = fusion_results[0.0]
print("選用 λ =", best_weight)

dev_evidence, dev_ranked = select_top(dev_scored, best_weight, EVIDENCE_TOP_K)
test_evidence, test_ranked = select_top(test_scored, best_weight, EVIDENCE_TOP_K)
evidence_recall_at5 = recall_at(dev_evidence, dev_df, as_sentences=True)
for k in (1, 3, 10):
    deeper, _ = select_top(dev_scored, best_weight, k)
    print(f"Evidence Recall@{k}: {recall_at(deeper, dev_df, as_sentences=True, k=k):.4f}")
print(f"Evidence Retrieval Recall@5: {evidence_recall_at5:.4f}")
print("測試證據：", test_evidence[0])

# 重排模型用完即釋放 GPU 記憶體，留給分類器。
del rerank_model
gc.collect()
torch.cuda.empty_cache()

重排推論耗時 863 秒
λ = 0.0（None 表示只用 bigram）: Evidence Recall@5 = 0.8465
λ = 1.0（None 表示只用 bigram）: Evidence Recall@5 = 0.8485
λ = 3.0（None 表示只用 bigram）: Evidence Recall@5 = 0.8485
λ = 10.0（None 表示只用 bigram）: Evidence Recall@5 = 0.8545
λ = None（None 表示只用 bigram）: Evidence Recall@5 = 0.7490
選用 λ = 10.0
Evidence Recall@1: 0.4780
Evidence Recall@3: 0.8020
Evidence Recall@10: 0.8720
Evidence Retrieval Recall@5: 0.8545
測試證據： [['沈懋孝', 1], ['沈懋孝', 0], ['明朝', 4], ['平民', 0], ['明朝', 0]]


## 4. Classification：RoBERTa-wwm-ext-large

輸入是 `[CLS] claim [SEP] 標題1：句子1；標題2：句子2；… [SEP]`，三類分別是 supports、refutes、NOT ENOUGH INFO。

訓練資料的證據要盡量貼近測試時的分布：
- supports／refutes：隨機取一組標準證據句，再用 bigram 排名補到 5 句，最後打亂順序；
- NOT ENOUGH INFO：沒有標準證據，直接用 bigram 排名前 5 句（與測試時一樣是「看起來相關但其實不足」的句子）。

訓練 2 個 epoch，使用線性 warmup、fp16 與 gradient accumulation；每個 epoch 結束時在 dev 上評估，保留最好的權重。

In [14]:
# [Generative AI] 本 cell 由 Claude 協助撰寫：組合分類器的輸入文字。
labels = ["supports", "refutes", "NOT ENOUGH INFO"]
label_to_id = {"SUPPORTS": 0, "REFUTES": 1, "NOT ENOUGH INFO": 2}


def join_context(items):
    return "；".join(sentence_input(page_id, text) for page_id, text in items)


rng = random.Random(SEED)
train_contexts = []
for row, candidates in zip(train_df.itertuples(), train_sentences):
    chosen = []
    if row.label != "NOT ENOUGH INFO":
        group = rng.choice(row.evidence)
        for item in group:
            text = page_lines.get(item[2], {}).get(item[3])
            if text and (item[2], text) not in chosen:
                chosen.append((item[2], text))
    for page_id, sid, text in bigram_rank(row.claim, candidates):
        if len(chosen) >= EVIDENCE_TOP_K:
            break
        if (page_id, text) not in chosen:
            chosen.append((page_id, text))
    rng.shuffle(chosen)
    train_contexts.append(join_context(chosen))

dev_contexts = [join_context([(p, t) for _, _, p, _, t in items]) for items in dev_ranked]
test_contexts = [join_context([(p, t) for _, _, p, _, t in items]) for items in test_ranked]
train_labels = train_df["label"].map(label_to_id).tolist()
dev_labels = dev_df["label"].map(label_to_id).tolist()
print(train_contexts[0][:300])

金朝：金朝作爲女真族所建的新興征服王朝，其部落制度的性質濃厚。；年輕人：年輕人泛指年紀輕的人類，相對於中年、老年。；金朝：按出虎亦曰按春，“按春曰金”，故國號名金，在部份文獻中，“金源”因此成爲金朝的代稱，亦有現代學者研究認爲，「金」實爲女真的漢譯，大金國意同「女真國」。；金朝：金朝在思想文化方面也逐漸趨向漢化，中期以後女真年輕人改漢姓、著漢服的現象普遍，金廷屢禁不止。；金朝：金朝，國號大金（女真文：-LSB- / amba - an antʃu - un / -RSB-；-LSB-ᠠᡳ᠌᠋ᠰᡳᠨᡤᡠᡵᡠᠨ, z = aisin gurun -RSB-；1115年- 1234年），是中國歷史上


In [15]:
# [Generative AI] 本 cell 由 Claude 協助撰寫：訓練 RoBERTa-wwm-ext-large 分類器。
set_seed(SEED)
cls_tokenizer = AutoTokenizer.from_pretrained(CLS_MODEL_NAME)
cls_model = AutoModelForSequenceClassification.from_pretrained(
    CLS_MODEL_NAME, num_labels=3,
    id2label=dict(enumerate(labels)), label2id={label: i for i, label in enumerate(labels)},
).to(device)

cls_train_loader = make_loader(cls_tokenizer, train_df["claim"].tolist(), train_contexts, train_labels,
                               CLS_BATCH_SIZE, CLS_MAX_LENGTH, shuffle=True, label_dtype=torch.long)
cls_dev_loader = make_loader(cls_tokenizer, dev_df["claim"].tolist(), dev_contexts, None,
                             64, CLS_MAX_LENGTH, shuffle=False, label_dtype=torch.long)
cls_test_loader = make_loader(cls_tokenizer, test_df["claim"].tolist(), test_contexts, None,
                              64, CLS_MAX_LENGTH, shuffle=False, label_dtype=torch.long)


def dev_accuracy():
    predicted = predict_logits(cls_model, cls_dev_loader).argmax(dim=1).tolist()
    return float(np.mean([p == g for p, g in zip(predicted, dev_labels)]))


start_time = time.time()
epoch_accuracies = train_model(cls_model, cls_train_loader, CLS_EPOCHS, CLS_LR, F.cross_entropy,
                               grad_accum=CLS_GRAD_ACCUM, eval_fn=dev_accuracy)
print(f"分類器訓練耗時 {time.time() - start_time:.0f} 秒，各 epoch dev accuracy：{epoch_accuracies}")

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/269k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.31GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: hfl/chinese-roberta-wwm-ext-large
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the ch

epoch 1 step 200/3002 loss 1.0928
epoch 1 step 400/3002 loss 1.0760
epoch 1 step 600/3002 loss 1.0683
epoch 1 step 800/3002 loss 1.0656
epoch 1 step 1000/3002 loss 1.0513
epoch 1 step 1200/3002 loss 1.0230
epoch 1 step 1400/3002 loss 0.9955
epoch 1 step 1600/3002 loss 0.9718
epoch 1 step 1800/3002 loss 0.9476
epoch 1 step 2000/3002 loss 0.9265
epoch 1 step 2200/3002 loss 0.9106
epoch 1 step 2400/3002 loss 0.8955
epoch 1 step 2600/3002 loss 0.8826
epoch 1 step 2800/3002 loss 0.8686
epoch 1 step 3000/3002 loss 0.8555
epoch 1 step 3002/3002 loss 0.8556
epoch 1 dev 分數：0.7113
epoch 2 step 200/3002 loss 0.5199
epoch 2 step 400/3002 loss 0.5337
epoch 2 step 600/3002 loss 0.5244
epoch 2 step 800/3002 loss 0.5322
epoch 2 step 1000/3002 loss 0.5277
epoch 2 step 1200/3002 loss 0.5278
epoch 2 step 1400/3002 loss 0.5306
epoch 2 step 1600/3002 loss 0.5291
epoch 2 step 1800/3002 loss 0.5278
epoch 2 step 2000/3002 loss 0.5228
epoch 2 step 2200/3002 loss 0.5231
epoch 2 step 2400/3002 loss 0.5230
epoch 

In [16]:
# [Generative AI] 本 cell 由 Claude 協助撰寫：預測並計算分類準確率與混淆矩陣。
dev_logits = predict_logits(cls_model, cls_dev_loader)
test_logits = predict_logits(cls_model, cls_test_loader)
dev_predictions = [labels[i] for i in dev_logits.argmax(dim=1).tolist()]
test_predictions = [labels[i] for i in test_logits.argmax(dim=1).tolist()]

classification_accuracy = float(np.mean([p.upper() == g for p, g in zip(dev_predictions, dev_df["label"])]))
print(f"Classification Accuracy: {classification_accuracy:.4f}")
confusion = pd.crosstab(dev_df["label"].rename("gold"),
                        pd.Series([p.upper() for p in dev_predictions], name="predicted"))
print(confusion)
print("測試預測分布：", Counter(test_predictions))


def cfever_score(predictions, evidences, df):
    hits = 0
    for prediction, evidence, row in zip(predictions, evidences, df.itertuples()):
        correct_label = prediction.upper() == row.label
        if row.label == "NOT ENOUGH INFO":
            hits += correct_label
            continue
        predicted_sentences = set(map(tuple, evidence[:5]))
        if correct_label and any(group <= predicted_sentences for group in gold_groups(row, as_sentences=True)):
            hits += 1
    return hits / len(df)

Classification Accuracy: 0.7113
predicted        NOT ENOUGH INFO  REFUTES  SUPPORTS
gold                                               
NOT ENOUGH INFO              553      270       177
REFUTES                       91      753       156
SUPPORTS                      58      114       828
測試預測分布： Counter({'supports': 2160, 'refutes': 2058, 'NOT ENOUGH INFO': 1782})


### NEI 校正：依 CFEVER Score 調整判斷

CFEVER Score 對 supports／refutes 題要求證據也正確。若證據沒找到，這題無論預測什麼標籤都得 0 分，
此時改判 NEI 不會損失分數，而且對真正的 NEI 題有幫助。因此加入兩個參數，並在 dev 上以 CFEVER Score 網格搜尋：

- `nei_bias`：加在 NEI logit 上的偏置；
- `evidence_threshold`：第 1 名證據的分數低於此門檻時，直接判 NEI。

In [17]:
# [Generative AI] 本 cell 由 Claude 協助撰寫：在 dev 上搜尋 NEI 偏置與證據分數門檻。
def top_evidence_score(ranked):
    if not ranked:
        return -1e9
    ce, bg = ranked[0][0], ranked[0][1]
    return bg if best_weight is None else ce + best_weight * bg


def calibrated(logits, ranked_lists, nei_bias, threshold):
    adjusted = logits.clone()
    adjusted[:, 2] += nei_bias
    output = []
    for label_id, ranked in zip(adjusted.argmax(dim=1).tolist(), ranked_lists):
        output.append("NOT ENOUGH INFO" if top_evidence_score(ranked) < threshold else labels[label_id])
    return output


dev_top_scores = np.array([top_evidence_score(r) for r in dev_ranked])
threshold_grid = [-1e10] + sorted(set(np.quantile(dev_top_scores, q) for q in np.linspace(0.02, 0.4, 20)))
uncalibrated_cfever = cfever_score(dev_predictions, dev_evidence, dev_df)
best = (uncalibrated_cfever, 0.0, -1e10)
for nei_bias in np.arange(-2.0, 3.01, 0.25):
    for threshold in threshold_grid:
        score = cfever_score(calibrated(dev_logits, dev_ranked, nei_bias, threshold), dev_evidence, dev_df)
        if score > best[0] + 1e-9:
            best = (score, float(nei_bias), float(threshold))
calibrated_cfever, NEI_BIAS, EVIDENCE_THRESHOLD = best
print(f"校正前 dev CFEVER Score: {uncalibrated_cfever:.4f}")
print(f"校正後 dev CFEVER Score: {calibrated_cfever:.4f}（nei_bias = {NEI_BIAS}, threshold = {EVIDENCE_THRESHOLD:.4f}）")

dev_predictions = calibrated(dev_logits, dev_ranked, NEI_BIAS, EVIDENCE_THRESHOLD)
test_predictions = calibrated(test_logits, test_ranked, NEI_BIAS, EVIDENCE_THRESHOLD)
calibrated_accuracy = float(np.mean([p.upper() == g for p, g in zip(dev_predictions, dev_df["label"])]))
print(f"校正後 Classification Accuracy: {calibrated_accuracy:.4f}")
print(pd.crosstab(dev_df["label"].rename("gold"), pd.Series([p.upper() for p in dev_predictions], name="predicted")))
print("校正後測試預測分布：", Counter(test_predictions))

校正前 dev CFEVER Score: 0.6557
校正後 dev CFEVER Score: 0.6907（nei_bias = 1.0, threshold = 0.9114）
校正後 Classification Accuracy: 0.7280
predicted        NOT ENOUGH INFO  REFUTES  SUPPORTS
gold                                               
NOT ENOUGH INFO              772      117       111
REFUTES                      235      636       129
SUPPORTS                     138       86       776
校正後測試預測分布： Counter({'NOT ENOUGH INFO': 2238, 'supports': 2018, 'refutes': 1744})


## 5. Validation：三階段彙整與 CFEVER Score

CFEVER Score：NEI 題只看標籤；supports／refutes 必須標籤正確，且前 5 句完整涵蓋任一標準證據組。

In [18]:
# [Generative AI] 本 cell 由 Claude 協助撰寫：彙整指標。
metrics = {
    "document_recall_at5": document_recall_at5,
    "document_candidate_upper_bound": candidate_upper_bound,
    "evidence_recall_at5_bigram": bigram_recall,
    "evidence_recall_at5_cross_encoder_only": cross_encoder_recall,
    "evidence_recall_at5_final": evidence_recall_at5,
    "classification_accuracy": classification_accuracy,
    "classification_accuracy_calibrated": calibrated_accuracy,
    "cfever_score_uncalibrated": uncalibrated_cfever,
    "cfever_score": cfever_score(dev_predictions, dev_evidence, dev_df),
}
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")
(OUTPUT_DIR / "dev_metrics.json").write_text(json.dumps(metrics, indent=2))

document_recall_at5: 0.8825
document_candidate_upper_bound: 0.8875
evidence_recall_at5_bigram: 0.7490
evidence_recall_at5_cross_encoder_only: 0.8465
evidence_recall_at5_final: 0.8545
classification_accuracy: 0.7113
classification_accuracy_calibrated: 0.7280
cfever_score_uncalibrated: 0.6557
cfever_score: 0.6907


394

## 6. 產生 submission.csv

欄位為 `id`、`predicted_label`、`predicted_evidence`；預測為 NEI 時證據填 `[]`。

In [19]:
# [Generative AI] 本 cell 由 Claude 協助撰寫：輸出提交檔。
evidence_strings = []
for label, evidence in zip(test_predictions, test_evidence):
    if label == "NOT ENOUGH INFO":
        evidence = []
    evidence_strings.append(json.dumps(evidence, ensure_ascii=False))

submission = pd.DataFrame({
    "id": test_df["id"],
    "predicted_label": test_predictions,
    "predicted_evidence": evidence_strings,
})
submission.to_csv(OUTPUT_DIR / "submission.csv", index=False, encoding="utf-8")
print("輸出筆數：", len(submission), "輸出位置：", OUTPUT_DIR / "submission.csv")
submission.head()

輸出筆數： 6000 輸出位置： /kaggle/working/submission.csv


,id,predicted_label,predicted_evidence
0,29579,NOT ENOUGH INFO,[]
1,3895,supports,"[[""鄱陽湖"", 0], [""鄱陽"", 3], [""江西省"", 1], [""鄱陽湖"", 2]..."
2,24522,refutes,"[[""赫伯特·胡佛"", 1], [""赫伯特·胡佛"", 0], [""胡佛"", 5], [""胡佛..."
3,11060,NOT ENOUGH INFO,[]
4,23607,supports,"[[""馬肯森級大巡洋艦"", 7], [""馬肯森級大巡洋艦"", 8], [""馬肯森級大巡洋艦""..."


# Report

Your Kaggle username: shiyu0318

Your public leaderboard score: 0.85600

Your private leaderboard score:


1. Describe your method for retrieving Wikipedia documents (5%) and report Recall@N on development set (5%).

Answer:
**方法：標題比對 + 啟發式排序。** 我發現 CFEVER 的 claim 是由 Wikipedia 句子改寫而來，證據頁面的標題通常直接出現在 claim 中。因此：
(1) 掃描完整 `wiki_pages.jsonl`（1,187,751 頁），建立「正規化標題 → 頁面」字典。正規化包括去掉 `_(消歧義說明)`、移除空白與底線、統一間隔號（例如「尼爾斯 · 索爾森」與「尼爾斯·索爾森」）。
(2) 另外從每頁開頭 300 字擷取別名（「又稱、簡稱、或譯、原名…」後面的詞），建立「別名 → 頁面」索引，處理 claim 使用別名或不同譯名的情況，共 117,773 個別名。
(3) 列舉 claim 中長度 1～30 的所有子字串，查標題與別名字典，得到候選頁面（dev 平均約 61.7 個）。
(4) 以線性分數排序。特徵包括：比對長度、該比對片段是否被更長的候選涵蓋（例如「芝加哥大學」涵蓋「芝加哥」）、claim 與頁面開頭 150 字及前 1000 字的字元 bigram 重疊率、是否為消歧義頁、是否只靠別名比對到。權重以 dev 網格搜尋決定。Recall 以前 5 頁計算，但第二階段實際從前 8 頁挑句子，提高證據候選的上限。
另外，助教範例只索引訓練集證據涉及的頁面，但 dev 的標準證據頁只有 2.5% 落在其中，所以我改為索引完整 Wikipedia。

| 設定（dev，只計 supports/refutes 共 2000 題） | Recall |
|---|---|
| 標題比對，未正規化，初版權重 | @5 = 0.8435 |
| + 標題正規化 | @5 = 0.8605 |
| + 權重網格搜尋 | @5 = 0.878 |
| + 別名索引（最終） | **@5 = 0.8825**，@1 = 0.8160，@3 = 0.8715，@8 = 0.8855，@10 = 0.8855 |
| 所有標題與別名候選的召回上限 | 0.8875 |

2. Describe your method for retrieving evidence (5%) and report Recall@N on development set  (5%).

Answer:
**方法：bigram 粗排 + cross-encoder 重排 + 分數融合。**
(1) 候選句為第一階段前 8 頁的所有句子，並保留 Wikipedia 原始句子編號。
(2) 基準方法以 claim 與句子的字元 bigram 重疊率排序，頁面第 0 句（定義句）略為加分。
(3) 重排模型是 `hfl/chinese-macbert-base` cross-encoder，輸入為 `[CLS] claim [SEP] 頁面標題：句子 [SEP]`，輸出一個相關分數。前面加上頁面標題，是因為許多證據句用代名詞指稱主題。訓練資料取自訓練集：正例為所有標準證據句；每題另取 5 個 bigram 排名高但非證據的困難負例，以及 3 個隨機負例（NEI 題只有負例）。損失為 Binary Cross Entropy，訓練 1 個 epoch，使用 fp16。
(4) 最終分數 = cross-encoder 分數 + λ × bigram 分數，λ 從 {0, 1, 3, 10, 只用 bigram} 中挑 dev Recall@5 最高者，取前 5 句。

| 方法（dev） | Evidence Recall@5 |
|---|---|
| Bigram 基準 | 0.7490 |
| Cross-encoder only（λ = 0） | 0.8465 |
| 融合 λ = 1 | 0.8485 |
| 融合 λ = 3 | 0.8485 |
| 融合（λ = 10，最終） | **0.8545**（@1 = 0.4780，@3 = 0.8020，@10 = 0.8720） |
| 候選句召回上限（受文件檢索限制） | 0.8855 |

3. Describe your classification method (5%) and report Accuracy on development set  (5%).

Answer:
**方法：`hfl/chinese-roberta-wwm-ext-large` 三分類。** 輸入為 `[CLS] claim [SEP] 標題1：句子1；…；標題5：句子5 [SEP]`，最長 256 token。
訓練資料使用全部 24,012 筆訓練資料，並盡量讓證據的分布接近測試時：supports/refutes 題隨機取一組標準證據，再用 bigram 排名補到 5 句後打亂順序；NEI 題沒有標準證據，因此直接用 bigram 前 5 句（與測試時「看似相關但資訊不足」的情況相同）。
超參數：AdamW，lr 1e-5，weight decay 0.01，10% 線性 warmup，等效 batch size 16（8 × gradient accumulation 2），2 個 epoch，fp16，gradient clipping 1.0。每個 epoch 結束在 dev 上評估，保留最好的權重。
推論時使用第二階段選出的前 5 句；預測為 NEI 時，提交的證據為 `[]`。

**NEI 校正：** CFEVER Score 對 supports/refutes 題同時要求證據正確。若證據沒有找到，這題無論預測哪個標籤都是 0 分，此時改判 NEI 不會損失分數，而且對真正的 NEI 題有幫助。因此我加入兩個參數：NEI logit 偏置 `nei_bias`，以及第 1 名證據分數的門檻（低於門檻直接判 NEI），在 dev 上以 CFEVER Score 網格搜尋。

| Dev | 校正前 | 校正後（nei_bias = 1.0，證據分數門檻 = 0.9114） |
|---|---|---|
| Classification Accuracy | **0.7113**（各 epoch：0.7113、0.6967，保留 epoch 1） | 0.7280 |
| CFEVER Score | 0.6557 | **0.6907** |

4. … Anything that can strengthen your report. (10%)

Answer:
**(a) 三階段與整體分數（dev 全部 3000 題）**

| 指標 | 助教範例（3000 訓練樣本、只索引訓練頁面） | 我的方法 |
|---|---|---|
| Document Recall@5 | ≤ 0.025（正確頁面只有 2.5% 在索引中） | 0.8825 |
| Evidence Recall@5 | ≤ 0.0065（正確句子只有 0.65% 在候選中） | 0.8545 |
| Classification Accuracy | 未測 | 0.7280 |
| CFEVER Score（校正後） | 未測 | 0.6907 |

**(b) 錯誤分析：文件檢索。** 在標題比對失敗的 dev 題目中，主要有三類：
1. 空白與符號不一致，例如「尼爾斯 · 索爾森」、`Super_Junior`。這類已透過正規化修正，Recall@5 提升約 1.7 個百分點。
2. 別名或譯名不同，例如「雷根」對「里根」、「威尼斯影展」對「威尼斯電影節」、「西洋棋」對「國際象棋」。從頁面開頭擷取別名修正了一部分，Recall@5 再提升約 0.45 個百分點。
3. 證據在 claim 沒有提到的頁面，需要多跳推理，例如 claim 提到「爬行動物」，證據卻在「羊膜動物」頁。
剩下約 11% 的題目屬於第 2、3 類，這是目前方法的上限（0.8875）。未來可以用 dense retrieval 或超連結擴展來改善。

**(c) 分類混淆矩陣（dev，列為標準答案、欄為預測）：**

| 校正前 | 預測 NEI | 預測 refutes | 預測 supports |
|---|---|---|---|
| NEI | 553 | 270 | 177 |
| refutes | 91 | 753 | 156 |
| supports | 58 | 114 | 828 |

| 校正後 | 預測 NEI | 預測 refutes | 預測 supports |
|---|---|---|---|
| NEI | 772 | 117 | 111 |
| refutes | 235 | 636 | 129 |
| supports | 138 | 86 | 776 |

校正前，模型最常把 NEI 誤判成 refutes（270 題）。原因是 NEI 題的證據是「看似相關」的句子，與 refutes 題中「相關但矛盾」的句子很像。校正後，NEI 的召回從 553 提升到 772，代價是部分 refutes 被改判為 NEI（91 → 235）。不過被改判的題目中，有不少本來就沒找到正確證據，原本就拿不到分，所以 CFEVER Score 整體提升 3.5 個百分點。另外，第 2 個 epoch 的 dev accuracy 下降（0.7113 → 0.6967），訓練 loss 卻持續下降，顯示 large 模型已開始過擬合，因此保留第 1 個 epoch 的權重。

**(d) 消融實驗：** 文件檢索的正規化、權重搜尋、別名索引效果見第 1 題表格；證據重排的 bigram、cross-encoder、融合三種設定比較見第 2 題表格；NEI 校正前後比較見第 3 題表格。

**(e) 誤差傳遞觀察：** CFEVER Score 要求 supports/refutes 題的證據完整命中，所以文件檢索的上限 0.8855 直接限制了這 2/3 題目的得分；證據重排後 Recall@5 = 0.8545，已接近這個上限。NEI 題只看標籤。因此改善文件檢索（尤其是別名對應與多跳）比單純換更大的分類模型更有效。

5. Generative AI Usage Statement

Answer:
我使用 Claude（Anthropic）協助完成本作業，包括：分析資料（發現範例程式的檢索語料覆蓋率問題、標題比對的覆蓋率）、撰寫與重構程式碼（標題索引、排序函式、cross-encoder 重排、分類器訓練流程）、在本機做驗證與權重網格搜尋，以及起草本報告文字。程式中每個由 AI 協助撰寫的 cell 都以 `# [Generative AI]` 註解標示。程式以助教提供的 sample code 為基礎修改；預訓練模型來自 Hugging Face：`hfl/chinese-macbert-base`、`hfl/chinese-roberta-wwm-ext-large`。實驗結果是我在 Kaggle 上實際執行取得，並經過我確認。